# Luma Phase 2 Module Implementation
This notebook contain the implementation of phase 2 of Luma Geospatial Engine

## Prerequisite
Earth engine initialization using service account

In [1]:
import ee 
import luma_ge
service_account_path = '../auth/python-ee-487304-3ab71cc11a87.json'
success = luma_ge.initialize_with_service_account(service_account_path)

if success:
    print("Earth Engine initialized with service account successfully!")
else:
    print("Service account initialization failed. Try to authenticate earth engine manually")

#Check authentication status
status = luma_ge.get_auth_status()
print(f"Initialized: {status['initialized']}")
print(f"Authenticated: {status['authenticated']}")
if status['project']:
    print(f"Project: {status['project']}")

Service account initialization failed: Please authorize access to your Earth Engine account by running

earthengine authenticate

in your command line, or ee.Authenticate() in Python, and then retry.


Earth Engine initialized with service account successfully!
Initialized: True
Authenticated: True


## Retrieve AOI from Earth Engine Asset Manager

In [ ]:
from luma_ge.data_acquisition import GEE_Asset_Manager
#Initialize the Asset Manager from Earth Engine
asset = GEE_Asset_Manager()
if asset.load_asset():
    regency_names = asset.get_regency_names()
    if regency_names:
        print(f"✓ Found {len(regency_names)} regencies")
        #use city name
        selected_regency = "Garut" 
        #Verify the city name exists in the regency list
        if selected_regency in regency_names:
            print(f"\n✓ Selected regency: {selected_regency}")
        else:
            print(f"\n✗ Regency '{selected_regency}' not found")
            print(f"\nAvailable regencies:")
            for i, name in enumerate(regency_names, 1):
                print(f"  {i}. {name}")
            selected_regency = None
    else:
        print("✗ No regency names found")
        selected_regency = None
else:
    print("✗ Failed to load asset")
    selected_regency = None
#Retrieve AOI geometry for the selected regency
if selected_regency:
    aoi = asset.get_regency_geometry(selected_regency)
    
    if aoi:
        print(f"✓ Successfully retrieved geometry for: {selected_regency}")
    else:
        print(f"✗ Failed to retrieve geometry for: {selected_regency}")
        aoi = None
else:
    print("✗ No regency selected")
    aoi = None

2026-06-03 10:28:45,880 - luma_ge.data_acquisition - INFO - Successfully loaded asset with 548 features
2026-06-03 10:28:46,290 - luma_ge.data_acquisition - INFO - Extracted 516 unique regency names


✓ Found 516 regencies

✓ Selected regency: Garut


2026-06-03 10:28:46,844 - luma_ge.data_acquisition - INFO - Successfully retrieved geometry for: Garut


✓ Successfully retrieved geometry for: Garut
  Bounds coordinates available


## New Feature: Sentinel-2 Data Retrieval



In [ ]:
from luma_ge.data_acquisition import Reflectance_Data, final_Image
import geemap
#initialize class
optical_reflectance = Reflectance_Data()
composite = final_Image()
#define the temporal range
start = '2025-01-01'
end = '2025-12-31'
#retrieve the sentinel 2 data
s2_data, metas2 = optical_reflectance.get_s2_optical_data(aoi, start, end, cloud_cover=20, compute_detailed_stats=False)
#final output: composite image
median_s2 = composite.get_temporal_composite(s2_data, aoi, calculate_coverage=False, coverage_scale=10)
#10m band
visnir = median_s2.select(['RED', 'GREEN', 'BLUE', 'NIR'])
#20m band
reswir = median_s2.select(['RED_EDGE1', 'RED_EDGE2', 'RED_EDGE3', 'RED_EDGE4', 'SWIR1', 'SWIR2'])
vis_10m = {'min': 0,'max': 0.3,'gamma': [0.95, 1.1, 1],'bands':['RED', 'GREEN', 'BLUE']}
res_20m = {'min': 0,'max': 0.3,'gamma': [0.95, 1.1, 1],'bands':['RED_EDGE1', 'RED_EDGE2', 'SWIR1']}
Map = geemap.Map()
Map.addLayer(median_s2, vis_10m, 'Sentinel-2 Composite 10 m band')
Map.addLayer(median_s2, res_20m, 'Sentinel-2 Composite 20 m band')
Map.centerObject(aoi, 9)
Map

2026-06-03 10:43:07,724 - Reflectance_Data - INFO - ReflectanceData initialized.
2026-06-03 10:43:07,725 - final_Image - INFO - final_Image creation initialized.
2026-06-03 10:43:07,725 - Reflectance_Data - INFO - Starting data fetch for Sentinel-2 Level-2A Surface Reflectance (Harmonized)
2026-06-03 10:43:07,725 - Reflectance_Data - INFO - Date range: 2025-01-01 to 2025-12-31
2026-06-03 10:43:07,725 - Reflectance_Data - INFO - Cloud cover threshold (image-level): 20%
2026-06-03 10:43:07,726 - Reflectance_Data - INFO - Cloud Score+ pixel threshold: 0.6
2026-06-03 10:43:07,726 - Reflectance_Data - INFO - Detailed statistics will not be computed
2026-06-03 10:43:07,726 - Reflectance_Stats - INFO - Reflectance Stats initialized.
2026-06-03 10:43:07,730 - Reflectance_Data - INFO - Filtered collection created (use compute_detailed_stats=True for more information)
2026-06-03 10:43:09,262 - final_Image - INFO - Creating median composite from 81 images
2026-06-03 10:43:09,264 - final_Image - I

Map(center=[-7.359399657147556, 107.78799735412501], controls=(WidgetControl(options=['position', 'transparent…